# 12e — Zernike Decomposition of DIA Difference Cutouts + Dipole Fit

## Scientific motivation

The Rubin DIA difference image (`Difference` stamp delivered via Fink alerts) for a
stable Gaia star should ideally be a zero image.  In practice it carries residual
structure whose **morphology encodes the physical origin of the artefact**:

| Zernike mode | Noll index | Physical meaning in DIA context |
|---|---|---|
| Piston      | Z1  | Monopole — overall flux offset (photometric zero-point error, template flux bias) |
| Tip / Tilt  | Z2, Z3 | **Dipole** — centroid shift between science and template (astrometric offset, rotation) |
| Defocus     | Z4  | Symmetric ring — PSF size mismatch (seeing difference) |
| Astigmatism | Z5, Z6 | **Quadrupole** — PSF ellipticity mismatch |
| Coma        | Z7, Z8 | Lateral asymmetry — higher-order PSF wing asymmetry |
| Trefoil     | Z9, Z10 | Higher-order PSF shape |
| Spherical aberration | Z11 | Radially symmetric high-order PSF |

Rubin AP uses Zernike moments internally to characterise PSF residuals; applying the
same decomposition to the alert-stream cutouts allows a **direct comparison** with the
AP pipeline diagnostics and gives a quantitative, visit-by-visit fingerprint of the
artefact type.

## Method

1. For each diaSource difference cutout `D(x,y)`:
   - Map the pixel grid onto the unit disk `ρ ∈ [0,1]`, `θ ∈ [0, 2π)`.
   - Pixels outside the inscribed disk are masked (set to NaN → excluded from the fit).
   - Project `D` onto the first `N_ZERN` Zernike polynomials (Noll convention) via
     a **weighted least-squares fit** (weight = 1/σ² per pixel if a noise estimate is
     available, else uniform).
   - Store the coefficient vector `c = [c1, c2, ..., cN]` and the residual image.

2. For each band:
   - **Subplot grid A** — coefficient spectrum `|c_j|` vs Noll index `j` for every
     visit, colour-coded by visit date, on shared axes.  Dominant modes are immediately
     visible.
   - **Subplot grid B** — per-visit bar chart of the first `N_DISPLAY` coefficients
     with amplitude and phase (angle of the complex reconstruction), one subplot per
     visit.
   - **Subplot grid C** — reconstructed image from Z1..Z_NDISP overlaid with the
     original difference cutout (both on the same diverging scale).

## Zernike convention

We use the **Noll (1976)** sequential indexing.  The polynomials are computed from
scratch using the standard radial / azimuthal recurrence — no external Zernike library
required, only `numpy` and `scipy`.

---
- **Author:** Sylvie Dagoret-Campagne — IJCLab / IN2P3 / CNRS — Université Paris-Saclay
- **Creation date:** 2026-05-15
- **Based on:** 12b_viewCutouts.ipynb
- **Subject:** Fink/LSST DIA — Zernike decomposition of difference-image cutouts


## 1. Imports & configuration

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec
from astropy.time import Time
from scipy.special import comb  # for Zernike radial polynomial

from scipy.optimize import least_squares

warnings.filterwarnings("ignore")
print(f"pandas {pd.__version__}  |  numpy {np.__version__}")

In [ ]:
%matplotlib inline

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# USER PARAMETERS  ← edit here
# ─────────────────────────────────────────────────────────────────────────────

objsid = {
    0: 170028527433285667,
    1: 170028526498480187,
    2: 170028485669027856,
    3: 313893022845108234,
    4: 170028526632697879,
    5: 170028529316003913,
}
DIAOBJECT_ID = objsid[1]

# Bands to analyse: None = all available, or e.g. ["r", "i"]
BANDS_FILTER = None

# Number of Zernike polynomials (Noll 1..N_ZERN).
# 15 covers up to radial order 4 (through spherical aberration).
# 21 covers up to radial order 5.
# Increase for richer morphological description (but need enough unmasked pixels).
N_ZERN = 15

# Number of Zernike terms to display in the per-visit bar-chart grid (Section 7)
N_DISPLAY = 11  # Z1..Z11

# Columns in the per-visit subplot grid (Sections 7 & 8)
NCOLS_GRID = 4

# Image type to decompose: 'Difference' (alert-stream DIA diff)
# Could also be 'Science' or 'Template' for comparison
CUTOUT_KIND = "Difference"

# Symmetric percentile for difference image display
DIFF_PERCENTILE = 99.0

# Save figures?
SAVE_FIGS = True
DIR_FIGS = "figs_FINK_BLOCK_LC_12d"

# ─────────────────────────────────────────────────────────────────────────────
# Fixed paths
# ─────────────────────────────────────────────────────────────────────────────
DIR_CUTOUTS = f"fullcutouts_{DIAOBJECT_ID}"
FILE_MANIFEST = os.path.join(DIR_CUTOUTS, "manifest.csv")
FILE_MANIFEST_FP = os.path.join(DIR_CUTOUTS, "manifest_fp.csv")
FILE_GAIA_MATCHED = os.path.join("data_FINK_BLOCK_LC_08b", "fink_diaobj_gaia_join_matched.csv")

AB_FLUX_ZERO = 3631e9
BAND_ORDER = list("ugrizy")
BAND_COLORS = {"u": "#9b59b6", "g": "#2ecc71", "r": "#e74c3c", "i": "#e67e22", "z": "#3498db", "y": "#795548"}

# Noll-index labels for the first 21 Zernike polynomials
ZERNIKE_LABELS = {
    1: "Z1\npiston",
    2: "Z2\ntip (x)",
    3: "Z3\ntilt (y)",
    4: "Z4\ndefocus",
    5: "Z5\nastg 45°",
    6: "Z6\nastg 0°",
    7: "Z7\ncoma x",
    8: "Z8\ncoma y",
    9: "Z9\ntrefoil",
    10: "Z10\ntrefoil",
    11: "Z11\nspher",
    12: "Z12",
    13: "Z13",
    14: "Z14",
    15: "Z15",
    16: "Z16",
    17: "Z17",
    18: "Z18",
    19: "Z19",
    20: "Z20",
    21: "Z21",
}

os.makedirs(DIR_FIGS, exist_ok=True)
print(f"diaObjectId : {DIAOBJECT_ID}")
print(f"Manifest    : {os.path.abspath(FILE_MANIFEST)}")
print(f"Figures dir : {os.path.abspath(DIR_FIGS)}")
print(f"N_ZERN={N_ZERN}   CUTOUT_KIND={CUTOUT_KIND}")

## 2. Zernike polynomial engine

We implement the **Noll (1976)** convention from scratch using only NumPy/SciPy.

Every polynomial $Z_j(\rho,\theta)$ is parameterised by radial order $n$ and
azimuthal frequency $m$ (with sign encoding the cos/sin parity):

$$
Z_j^{m>0} = \sqrt{2(n+1)}\,R_n^m(\rho)\cos(m\theta)
\qquad
Z_j^{m<0} = \sqrt{2(n+1)}\,R_n^{|m|}(\rho)\sin(|m|\theta)
\qquad
Z_j^{m=0} = \sqrt{n+1}\,R_n^0(\rho)
$$

where the radial polynomial is
$$
R_n^m(\rho) = \sum_{s=0}^{(n-m)/2}
\frac{(-1)^s (n-s)!}{s!\,[(n+m)/2-s]!\,[(n-m)/2-s]!}\,\rho^{n-2s}
$$

The polynomials are **orthonormal on the unit disk**:
$\langle Z_i, Z_j\rangle = \delta_{ij}$.

In [ ]:
from math import factorial


def noll_to_nm_bad_claude(j):
    """Convert Noll sequential index j (1-based) to (n, m) with Noll sign convention.

    Returns
    -------
    n : int   radial order  (n >= 0)
    m : int   azimuthal frequency  (|m| <= n, same parity as n)
              positive m → cos term, negative m → sin term
    """
    n = 0
    j_remaining = j
    while j_remaining > n + 1:
        n += 1
        j_remaining -= n
    # j_remaining is now the 1-based position within radial order n
    # Noll ordering within each n: m = 0, ±1, ±2, ...
    # Rebuild from Noll's explicit formula
    # Use cumulative offset approach
    n = int(np.ceil((-3 + np.sqrt(9 + 8 * (j - 1))) / 2))
    offset = n * (n + 2) // 2 - n  # first Noll index of radial order n is offset+1
    # position p within radial order n (0-based)
    # Re-derive cleanly
    n = 0
    cumul = 0
    while cumul + n + 1 < j:
        n += 1
        cumul += n
    # Position within this radial shell (0-based)
    k = j - cumul - 1  # 0-based index within shell of size n+1
    # Noll assigns m values in this order within shell n:
    # k=0 → m=0  (only when n even and k=0)
    # then pairs (m>0, m<0) or (m<0, m>0) depending on j parity
    # Simpler closed-form:
    # m_abs goes 0, 1, 1, 2, 2, ... within shell, with parity rule
    # Use the standard lookup derived from Noll 1976 Table 1
    m_abs = (k + (n % 2)) // 2 * 2 - (n % 2) * (1 - (k % 2))
    # Sign: j even → positive (cos), j odd → negative (sin) for m != 0
    if m_abs == 0:
        m = 0
    elif j % 2 == 0:
        m = m_abs
    else:
        m = -m_abs
    return n, m

In [ ]:
def noll_to_nm_bad_chatgpt(j):
    """Convert Noll index j (1-based) to (n, m)."""

    # --- find n (radial order) ---
    n = int(np.ceil((-3 + np.sqrt(9 + 8 * (j - 1))) / 2))

    # index of first mode with this n
    j0 = n * (n + 1) // 2 + 1

    # position inside this n-shell
    k = j - j0

    # --- compute m ---
    if n % 2 == 0:
        m = 2 * (k // 2)
    else:
        m = 2 * (k // 2) + 1

    # apply Noll sign convention
    if k % 2 == 1:
        m = -m

    return n, m

In [ ]:
# Final correction from chatGPT
def noll_to_nm(j):
    """Convert Noll index j (1-based) to (n, m) following Noll (1976). according chatgpt"""

    if j < 1:
        raise ValueError("j must be >= 1")

    count = 1
    n = 0

    while True:
        # build the correct |m| sequence for this n
        if n % 2 == 0:
            m_list = [0]
            for m in range(2, n + 1, 2):
                m_list.extend([m, m])
        else:
            m_list = []
            for m in range(1, n + 1, 2):
                m_list.extend([m, m])

        for m_abs in m_list:
            if count == j:
                # assign sign (Noll convention)
                if m_abs == 0:
                    return n, 0
                elif count % 2 == 0:
                    return n, m_abs
                else:
                    return n, -m_abs
            count += 1

        n += 1

In [ ]:
# Validate against Noll 1976 Table 1
_NOLL_EXPECTED = {
    1: (0, 0),
    2: (1, 1),
    3: (1, -1),
    4: (2, 0),
    5: (2, -2),
    6: (2, 2),
    7: (3, -1),
    8: (3, 1),
    9: (3, -3),
    10: (3, 3),
    11: (4, 0),
    12: (4, 2),
}


for _j, _expected in _NOLL_EXPECTED.items():
    _got = noll_to_nm(_j)
    assert _got == _expected, f"noll_to_nm({_j}) = {_got}, expected {_expected}"
print("Noll index conversion validated against Table 1.")


def zernike_radial(n, m_abs, rho):
    """Evaluate the radial Zernike polynomial R_n^m(rho) on array rho.

    Uses the explicit summation formula (safe for small n, numerically
    stable up to n~20).
    """
    m = m_abs
    assert (n - m) % 2 == 0, f"n={n}, m={m} violates parity rule"
    result = np.zeros_like(rho, dtype=np.float64)
    for s in range((n - m) // 2 + 1):
        coeff = (
            (-1) ** s
            * factorial(n - s)
            / (factorial(s) * factorial((n + m) // 2 - s) * factorial((n - m) // 2 - s))
        )
        result += coeff * rho ** (n - 2 * s)
    return result


def zernike_poly(j, rho, theta):
    """Evaluate the j-th Zernike polynomial (Noll, orthonormal on unit disk).

    Parameters
    ----------
    j     : int    Noll index (1-based)
    rho   : array  normalised radius in [0, 1]  (pixels outside disk → undefined)
    theta : array  azimuthal angle in radians, same shape as rho

    Returns
    -------
    Z : array  same shape as rho
    """
    n, m = noll_to_nm(j)
    R = zernike_radial(n, abs(m), rho)
    if m == 0:
        norm = np.sqrt(n + 1)
        return norm * R
    elif m > 0:
        norm = np.sqrt(2 * (n + 1))
        return norm * R * np.cos(m * theta)
    else:  # m < 0
        norm = np.sqrt(2 * (n + 1))
        return norm * R * np.sin(abs(m) * theta)


print("Zernike polynomial engine ready.")

## 3. Build the Zernike basis matrix on a pixel grid

The basis is built **once per image shape** and cached, so images of the same
size reuse the precomputed basis.  Pixels outside the inscribed unit disk are
masked out.

In [ ]:
# Cache: {(ny, nx, n_zern): (basis_matrix, disk_mask, rho, theta)}
_ZERNIKE_CACHE = {}


def build_zernike_basis(ny, nx, n_zern):
    """Build the Zernike design matrix for a (ny, nx) pixel grid.

    The pixel coordinate system is centred on the image centre.
    The inscribed disk has radius = min(ny, nx) / 2 pixels → rho_max = 1.
    Pixels outside the disk are masked.

    Returns
    -------
    A        : (n_in_disk, n_zern) array  — design matrix (unmasked pixels × polynomials)
    mask     : (ny, nx) bool array        — True where pixel is inside the disk
    rho      : (ny, nx) array             — normalised radius
    theta    : (ny, nx) array             — azimuthal angle [rad]
    """
    key = (ny, nx, n_zern)
    if key in _ZERNIKE_CACHE:
        return _ZERNIKE_CACHE[key]

    # Pixel coordinates centred on image centre
    cy, cx = (ny - 1) / 2.0, (nx - 1) / 2.0
    yy, xx = np.mgrid[0:ny, 0:nx]
    dy = yy - cy
    dx = xx - cx

    # Normalise so that the inscribed disk has radius 1
    r_max = min(ny, nx) / 2.0
    rho = np.sqrt(dx**2 + dy**2) / r_max
    # theta = np.arctan2(dy, dx)   # standard angle convention
    theta = np.arctan2(-dy, dx)  # try to correct for a bad orientation, but it does not improve the fit

    # Disk mask: keep pixels strictly inside (or on the boundary of) the unit disk
    mask = rho <= 1.0
    n_in = int(mask.sum())

    rho_in = rho[mask]
    theta_in = theta[mask]

    # Design matrix: columns = Zernike polynomials evaluated at each in-disk pixel
    A = np.zeros((n_in, n_zern), dtype=np.float64)
    for j in range(1, n_zern + 1):
        A[:, j - 1] = zernike_poly(j, rho_in, theta_in)

    _ZERNIKE_CACHE[key] = (A, mask, rho, theta)
    print(f"  Built Zernike basis: shape={ny}×{nx}  n_in_disk={n_in}  n_zern={n_zern}")
    return A, mask, rho, theta


print("Zernike basis builder ready (cached per image shape).")

### Test Zernike polynoms

In [ ]:
nx = 30
ny = 30
A, mask, rho, theta = build_zernike_basis(ny, nx, N_ZERN)

Z1 = zernike_poly(1, rho[mask], theta[mask])
Z2 = zernike_poly(2, rho[mask], theta[mask])
Z3 = zernike_poly(3, rho[mask], theta[mask])

In [ ]:
Z2_img = np.zeros_like(rho)
Z2_img[mask] = Z2

Z3_img = np.zeros_like(rho)
Z3_img[mask] = Z3

### Check dipole orientation
- Z2 gradient croissant vers axe X :

        gauche → négatif
        droite → positif


- Z3 gradient croissant vers axe Y (avec origin = lower):

        bas → positif
        haut → negatif

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 5), constrained_layout=True)

im1 = axs[0].imshow(Z2_img, origin="lower")
axs[0].set_title("Z2 (dipole)")
fig.colorbar(im1, ax=axs[0], fraction=0.046, pad=0.04)

im2 = axs[1].imshow(Z3_img, origin="lower")
axs[1].set_title("Z3 (dipole)")
fig.colorbar(im2, ax=axs[1], fraction=0.046, pad=0.04)

plt.show()

## 4. Projection and reconstruction functions

In [ ]:
def fit_zernike(image, n_zern, weights=None):
    """Project a 2-D image onto the first n_zern Zernike polynomials.

    The fit is performed on pixels inside the inscribed unit disk only.
    NaN pixels inside the disk are excluded from the fit.

    Parameters
    ----------
    image   : (ny, nx) float32 array
    n_zern  : int
    weights : (ny, nx) array or None
        Per-pixel weights (e.g. 1/σ²).  If None, uniform weights are used.

    Returns
    -------
    coeffs     : (n_zern,) array  — Zernike coefficients c_j
    recon      : (ny, nx) array  — reconstructed image (in-disk pixels only; NaN outside)
    residual   : (ny, nx) array  — image − reconstruction
    fit_quality: dict with 'rms_input', 'rms_residual', 'r2' (variance explained)
    """
    ny, nx = image.shape
    A, disk_mask, rho, theta = build_zernike_basis(ny, nx, n_zern)

    # Extract in-disk pixels
    img_in = image[disk_mask].astype(np.float64)

    # Build combined validity mask: disk AND finite
    finite_mask_2d = np.isfinite(image) & disk_mask
    valid_in_disk = finite_mask_2d[disk_mask]  # boolean, size = n_in_disk

    A_fit = A[valid_in_disk, :]
    img_fit = img_in[valid_in_disk]

    # Proposed by claude-desktop 2026-05-15
    if weights is not None:
        w_in = weights[disk_mask][valid_in_disk].astype(np.float64)
        w_in = np.maximum(w_in, 0.0)  # ensure non-negative
        W = np.diag(w_in)
        # Weighted least-squares: (A^T W A) c = A^T W b
        ATA = A_fit.T @ W @ A_fit
        ATb = A_fit.T @ (w_in * img_fit)
    else:
        ATA = A_fit.T @ A_fit
        ATb = A_fit.T @ img_fit

    # Solve with least-squares (pinv for numerical safety) proposed by claude
    # coeffs, *_ = np.linalg.lstsq(ATA, ATb, rcond=None)

    # Proposed by chatGPT 2026-05-15
    if weights is None:
        coeffs, *_ = np.linalg.lstsq(A_fit, img_fit, rcond=None)
    else:
        w_in = weights[disk_mask][valid_in_disk].astype(np.float64)
        sqrt_w = np.sqrt(w_in)
        A_w = A_fit * sqrt_w[:, None]
        y_w = img_fit * sqrt_w
        coeffs, *_ = np.linalg.lstsq(A_w, y_w, rcond=None)

    # Reconstruct on the full in-disk grid (all pixels, including those excluded from fit)
    recon_in = A @ coeffs  # shape (n_in_disk,)
    recon_2d = np.full((ny, nx), np.nan, dtype=np.float64)
    recon_2d[disk_mask] = recon_in

    residual_2d = image.astype(np.float64) - recon_2d

    # Fit quality on the valid pixels
    img_valid = img_fit
    recon_valid = A_fit @ coeffs
    resid_valid = img_valid - recon_valid
    rms_input = float(np.sqrt(np.mean(img_valid**2)))
    rms_residual = float(np.sqrt(np.mean(resid_valid**2)))
    ss_tot = float(np.sum((img_valid - np.mean(img_valid)) ** 2))
    ss_res = float(np.sum(resid_valid**2))
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan

    fit_quality = {
        "rms_input": rms_input,
        "rms_residual": rms_residual,
        "r2": r2,
        "n_fit_pixels": int(valid_in_disk.sum()),
    }
    return coeffs, recon_2d.astype(np.float32), residual_2d.astype(np.float32), fit_quality


def zernike_group_power(coeffs):
    """Compute the power per morphological group.

    Returns a dict with keys: 'monopole', 'dipole', 'quadrupole',
    'defocus', 'coma', 'trefoil', 'spherical', 'higher'.
    Power = sum of squared coefficients in the group.
    """
    groups = {
        "Z1_piston": [1],
        "Z2-3_dipole": [2, 3],
        "Z4_defocus": [4],
        "Z5-6_astig": [5, 6],
        "Z7-8_coma": [7, 8],
        "Z9-10_trefoil": [9, 10],
        "Z11_spher": [11],
        "Z12+_higher": list(range(12, len(coeffs) + 1)),
    }
    power = {}
    for name, indices in groups.items():
        valid = [i for i in indices if i <= len(coeffs)]
        power[name] = float(np.sum(coeffs[np.array(valid) - 1] ** 2)) if valid else 0.0
    return power


print("Fit / reconstruction functions ready.")

## Test of Zernike polynom fit

🧭 1. **Définition correcte de A**

Ta matrice de design est :

$$A_{i,j} = Z_j(\rho_i, \theta_i)$$
- i = pixel dans le disque (mask)
- j = indice Zernike
- $Z_j$​ = polynôme de Zernike évalué au pixel
👉 donc :
- chaque **colonne = un mode Zernike**
- chaque **ligne = un pixel**


🧪 2. **Comment construire A (version correcte)**

Tu as déjà presque tout. Le cœur est :
```python
rho_in = rho[mask]
theta_in = theta[mask]
```

✔️ construction de A

```python
A = np.zeros((rho_in.size, n_zern), dtype=np.float64)

for j in range(1, n_zern + 1):
    A[:, j-1] = zernike_poly(j, rho_in, theta_in)
```

🧠 3. **Interprétation simple**

| objet | shape         | signification |
| ----- | ------------- | ------------- |
| `A`   | (Npix, Nzern) | base Zernike  |
| `y`   | (Npix,)       | image aplatie |
| `c`   | (Nzern,)      | coefficients  |


⚠️ 4. **point critique (très important dans ton cas)**

Tu dois absolument garantir :

✔️ même masque partout

```python
assert A.shape[0] == np.sum(mask)
assert len(y) == np.sum(mask)
```

✔️ correct

```python
y = image[mask]
```

🧪 6. **fit complet correct**

Voici pipeline propre :

```python
A, mask, rho, theta = build_zernike_basis(ny, nx, n_zern)
y = image[mask]
c, *_ = np.linalg.lstsq(A, y, rcond=None)
y_fit = A @ c
res = y - y_fit
```


📊 7. **diagnostic immédiat**

```python
print("rel error:", np.linalg.norm(res) / np.linalg.norm(y))
```

🔥 8. test sanity (OBLIGATOIRE)

```python
test = 2 * zernike_poly(2, rho_in, theta_in)
c = np.linalg.lstsq(A, test, rcond=None)[0]
print(c[:5])
```

👉 attendu :

pic net sur j=2 uniquement

🎯 10. **résumé ultra clair**

- ✔️ A se calcule comme :

```python
A[i,j] = zernike_poly(j, rho[mask][i], theta[mask][i])
```
- ✔️ y doit être :

```python
y = image[mask]
```
- ✔️ fit :

```python
c = np.linalg.lstsq(A, y)
```

### Construct the image to fit

In [ ]:
Z_true = 3 * Z2_img + -1.5 * Z3_img

#### Check the image to fit

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(4, 4), constrained_layout=True)

im = ax.imshow(Z_true, origin="lower")
ax.set_title("3 * Z2_img + -1.5 * Z3_img(dipole)")
fig.colorbar(im1, ax=axs[0], fraction=0.046, pad=0.04)

#### Build the model

ÉTAPE 1 — construire la base A 

⚠️ IMPORTANT : A doit être construit sur rho/theta + mask

In [ ]:
A, mask, rho, theta = build_zernike_basis(ny, nx, N_ZERN)

In [ ]:
A.shape

In [ ]:
mask.shape

In [ ]:
theta.shape

#### Build the data vector

🧭 ÉTAPE 2 — construire le vecteur d’observation y

In [ ]:
nx * ny

In [ ]:
y = Z_true[mask]

In [ ]:
y.shape

#### Fit
🧭 ÉTAPE 3 — fit

In [ ]:
coeff, *_ = np.linalg.lstsq(A, y, rcond=None)
y_fit = A @ coeff

In [ ]:
coeff.shape

In [ ]:
y_fit.shape

#### Residuals

In [ ]:
res = y - y_fit
print("relative error =", np.linalg.norm(res) / np.linalg.norm(y))
print("coeff =", coeff[:10])

In [ ]:
res.shape

#### Retransform in images

In [ ]:
Z_fit_img = np.zeros_like(rho)
Z_fit_img[mask] = y_fit

In [ ]:
Z_res_img = np.zeros_like(rho)
Z_res_img[mask] = res

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)
im1 = ax1.imshow(Z_true, origin="lower")
im2 = ax2.imshow(Z_fit_img, origin="lower")
im3 = ax3.imshow(Z_res_img, origin="lower")
plt.suptitle(" Fit of 3 * Z2_img + -1.5 * Z3_img(dipole)")
ax1.set_title("original image")
ax2.set_title("fitted image")
ax3.set_title("res image")
fig.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04)
fig.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)
fig.colorbar(im3, ax=ax3, fraction=0.046, pad=0.04)

#### Debug fit_zernike

In [ ]:
coeffs, recon, residual, quality = fit_zernike(Z_true, N_ZERN)
power = zernike_group_power(coeffs)

In [ ]:
y_fit = A @ coeff

In [ ]:
res = y - y_fit
print("relative error =", np.linalg.norm(res) / np.linalg.norm(y))
print("coeff =", coeff[:10])

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)
im1 = ax1.imshow(Z_true, origin="lower")
im2 = ax2.imshow(Z_fit_img, origin="lower")
im3 = ax3.imshow(Z_res_img, origin="lower")
plt.suptitle(" Fit with fit_zernike() ")
ax1.set_title("original image")
ax2.set_title("fitted image")
ax3.set_title("res image")
fig.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04)
fig.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)
fig.colorbar(im3, ax=ax3, fraction=0.046, pad=0.04)

- we obtain the same result 

- However the residuals are badly interpreted a residuals

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
im1 = ax1.imshow(recon, origin="lower")
fig.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04)

im2 = ax2.imshow(residual, origin="lower")
fig.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)

ax1.set_title("recon2d (fit)")
ax2.set_title("residuals")
plt.show()

## Dipole model fit functions

In [ ]:
def dipole_model_v0(params, x, y, sigma):
    x0, y0, dx, dy, A, B = params

    g1 = np.exp(-((x - (x0 + dx)) ** 2 + (y - (y0 + dy)) ** 2) / (2 * sigma**2))
    g2 = np.exp(-((x - (x0 - dx)) ** 2 + (y - (y0 - dy)) ** 2) / (2 * sigma**2))

    return A * (g1 - g2) + B


def dipole_model(params, x, y, sigma):
    x0, y0, sep, theta, A, B = params

    dx = sep * np.cos(theta)
    dy = sep * np.sin(theta)

    g1 = np.exp(-((x - (x0 + dx)) ** 2 + (y - (y0 + dy)) ** 2) / (2 * sigma**2))
    g2 = np.exp(-((x - (x0 - dx)) ** 2 + (y - (y0 - dy)) ** 2) / (2 * sigma**2))

    return A * (g1 - g2) + B


def fit_dipole_analytic(image, sigma=1.5):
    ny, nx = image.shape
    y, x = np.indices((ny, nx))

    valid = np.isfinite(image)
    x_data = x[valid]
    y_data = y[valid]
    z_data = image[valid]

    # ---- initial guess ----
    x0 = np.mean(x_data)
    y0 = np.mean(y_data)

    gx = np.gradient(image, axis=1)
    gy = np.gradient(image, axis=0)

    dx0 = np.mean(gx[valid])
    dy0 = np.mean(gy[valid])
    norm = np.hypot(dx0, dy0) + 1e-8
    dx0 /= norm
    dy0 /= norm

    A0 = np.nanmax(image) - np.nanmin(image)
    B0 = np.nanmedian(image)

    p0 = [x0, y0, dx0, dy0, A0, B0]

    def residuals(p):
        return dipole_model(p, x_data, y_data, sigma) - z_data

    res = least_squares(residuals, p0, loss="soft_l1")

    p_opt = res.x

    model_full = dipole_model(p_opt, x, y, sigma)
    residual_full = image - model_full

    # metrics
    rms_in = np.sqrt(np.mean(z_data**2))
    rms_res = np.sqrt(np.mean(res.fun**2))
    ss_tot = np.sum((z_data - np.mean(z_data)) ** 2)
    ss_res = np.sum(res.fun**2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan

    quality = {
        "rms_input": float(rms_in),
        "rms_residual": float(rms_res),
        "r2": float(r2),
    }

    return p_opt, model_full, residual_full, quality

In [ ]:
def fit_dipole_linear(image, sigma=1.5):
    ny, nx = image.shape
    y, x = np.indices((ny, nx))

    valid = np.isfinite(image)

    x0 = np.mean(x[valid])
    y0 = np.mean(y[valid])

    # Gaussian center
    G = np.exp(-((x - x0) ** 2 + (y - y0) ** 2) / (2 * sigma**2))

    dGdx = -(x - x0) / sigma**2 * G
    dGdy = -(y - y0) / sigma**2 * G

    A_mat = np.stack([dGdx[valid], dGdy[valid], np.ones(np.sum(valid))], axis=1)
    b = image[valid]

    coeffs, *_ = np.linalg.lstsq(A_mat, b, rcond=None)

    Ax, Ay, B = coeffs

    model = Ax * dGdx + Ay * dGdy + B
    residual = image - model

    residual_vec = b - A_mat @ coeffs

    r2 = 1 - np.sum(residual_vec**2) / np.sum((b - np.mean(b)) ** 2)

    rms_in = np.sqrt(np.mean(b**2))
    rms_res = np.sqrt(np.mean(residual_vec**2))

    quality = {
        "r2": float(r2),
        "rms_input": float(rms_in),
        "rms_residual": float(rms_res),
    }

    return coeffs, model, residual, quality

## 5. Utility functions (shared with 12b)

In [ ]:
def mjd_to_date(mjd):
    """MJD (TAI) → ISO date string YYYY-MM-DD."""
    try:
        return Time(float(mjd), format="mjd", scale="tai").isot[:10]
    except Exception:
        return "?"


def parse_bool(val):
    """Coerce a dipole flag to Python bool."""
    if isinstance(val, bool):
        return val
    if isinstance(val, (int, float)) and np.isfinite(float(val)):
        return bool(int(val))
    if isinstance(val, str):
        return val.strip().lower() in ("true", "1", "yes")
    return False


def load_npy(src_id, band, kind):
    """Load fullcutouts_{obj}/cutouts/{src_id}_{band}_{kind}.npy → float32 or None."""
    fpath = os.path.join(DIR_CUTOUTS, "cutouts", f"{src_id}_{band}_{kind}.npy")
    return np.load(fpath).astype(np.float32) if os.path.exists(fpath) else None


def zscale(arr, lo=1.0, hi=99.0):
    """Percentile z-scale ignoring NaNs."""
    finite = arr[np.isfinite(arr)]
    return (
        (0.0, 1.0)
        if len(finite) == 0
        else (float(np.percentile(finite, lo)), float(np.percentile(finite, hi)))
    )


def symvlim(arr, percentile=99.0):
    """Symmetric half-range for diverging colour scale."""
    finite = arr[np.isfinite(arr)]
    return max(float(np.percentile(np.abs(finite), percentile)), 1e-9) if len(finite) else 1e-9


def draw_crosshair(ax, shape):
    ny, nx = shape
    ax.axvline(nx / 2, color="yellow", lw=0.8, ls="--", alpha=0.7)
    ax.axhline(ny / 2, color="yellow", lw=0.8, ls="--", alpha=0.7)


print("Utility functions ready.")

## 6. Load manifest & Gaia metadata

In [ ]:
if not os.path.exists(FILE_MANIFEST):
    raise FileNotFoundError(
        f"{FILE_MANIFEST} not found.\nRun: python fink_download_full_cutouts.py --obj_id {DIAOBJECT_ID}"
    )

df = pd.read_csv(FILE_MANIFEST)
df["isDipole"] = df["r:isDipole"].fillna(False).apply(parse_bool)
df = df.sort_values("r:midpointMjdTai").reset_index(drop=True)
print(f"Manifest : {len(df)} diaSources   bands: {sorted(df['r:band'].unique())}")

# Gaia metadata
gaia_G_mag, gaia_status, fink_group = np.nan, "?", "?"
if os.path.exists(FILE_GAIA_MATCHED):
    dg = pd.read_csv(FILE_GAIA_MATCHED)
    hit = dg[dg["diaObjectId"].astype(str) == str(DIAOBJECT_ID)]
    if len(hit):
        r = hit.iloc[0]
        gaia_G_mag = float(r.get("gaia_phot_g_mean_mag", np.nan))
        gaia_status = str(r.get("gaia_status", "?"))
        fink_group = str(r.get("group", "?"))
G_str = f"{gaia_G_mag:.2f}" if np.isfinite(gaia_G_mag) else "?"
print(f"Gaia G={G_str}  status={gaia_status}  group={fink_group}")

bands_available = [b for b in BAND_ORDER if b in df["r:band"].values]
bands_to_use = [b for b in bands_available if BANDS_FILTER is None or b in BANDS_FILTER]
print(f"Bands to analyse : {bands_to_use}")

## 7. Run Zernike decomposition — collect all coefficients

In [ ]:
# results[band] = list of dicts, one per valid diaSource
results = {}

# Loop on bands
# ==============
for band in bands_to_use:
    df_b = df[df["r:band"] == band].sort_values("r:midpointMjdTai").reset_index(drop=True)
    print(f"\n{'─' * 60}")
    print(f"  Band {band}  —  {len(df_b)} diaSources")
    print(f"{'─' * 60}")

    band_results = []
    for src_idx, (_, row) in enumerate(df_b.iterrows(), start=1):
        src_id = str(row["r:diaSourceId"])
        visit = row.get("r:visit", "?")
        mjd = float(row["r:midpointMjdTai"])
        date = mjd_to_date(mjd)
        is_dip = bool(row["isDipole"])
        psf_f = float(row.get("r:psfFlux", np.nan))
        snr = float(row.get("r:snr", np.nan))

        arr = load_npy(src_id, band, CUTOUT_KIND)
        if arr is None:
            print(f"  [{src_idx:02d}] visit={visit}  {date}  → {CUTOUT_KIND} missing, skip")
            continue

        # =========================
        # Fit Zernike (on garde)
        # =========================
        coeffs_z, recon_z, residual_z, quality_z = fit_zernike(arr, N_ZERN)
        power_z = zernike_group_power(coeffs)

        # Dipole amplitude: |c2| + |c3| (tip + tilt)
        dip_amp_z = float(np.sqrt(coeffs[1] ** 2 + coeffs[2] ** 2)) if N_ZERN >= 3 else np.nan
        # Dipole angle (radians, measured from x-axis)
        dip_angle_z = float(np.arctan2(coeffs[2], coeffs[1])) * 180 / np.pi if N_ZERN >= 3 else np.nan

        # =========================
        # Fit dipôle analytique
        # =========================
        p_opt, recon_d, residual_d, quality_d = fit_dipole_analytic(arr)

        x0, y0, dx, dy, A, B = p_opt
        dip_amp_d = float(A)
        dip_angle_d = float(np.degrees(np.arctan2(dy, dx)))
        dip_sep_d = float(2 * np.hypot(dx, dy))

        # =========================
        # Fit dipole linéaire (RECOMMANDÉ)
        # =========================
        lin_coeffs, model_lin, residual_lin, quality_lin = fit_dipole_linear(arr, sigma=1.5)

        Ax, Ay, B_lin = lin_coeffs
        dip_amp_lin = float(np.hypot(Ax, Ay))
        dip_angle_lin = float(np.degrees(np.arctan2(Ay, Ax)))

        band_results.append(
            {
                "src_id": src_id,
                "visit": visit,
                "mjd": mjd,
                "date": date,
                "is_dipole": is_dip,
                "psfFlux": psf_f,
                "snr": snr,
                "arr": arr,
                # =================
                # Zernike
                # =================
                "recon_z": recon_z,
                "residual_z": residual_z,
                "z_coeffs": coeffs_z,
                "z_power": power_z,
                "dip_amp_z": dip_amp_z,
                "dip_angle_z": dip_angle_z,
                "quality_z": quality_z,
                # =================
                # Dipole nonlinear
                # =================
                "recon_d_nl": recon_d,
                "residual_d_nl": residual_d,
                "dip_amp_d_nl": dip_amp_d,
                "dip_angle_d_nl": dip_angle_d,
                "dip_sep_d_nl": dip_sep_d,
                "quality_d_nl": quality_d,
                # =================
                # Dipole linear (stable)
                # =================
                "recon_d_lin": model_lin,
                "residual_d_lin": residual_lin,
                "dip_amp_d_lin": dip_amp_lin,
                "dip_angle_d_lin": dip_angle_lin,
                "quality_d_lin": quality_lin,
            }
        )
        print(
            f"  [{src_idx:02d}] visit={visit}  {date}  shape={arr.shape}\n"
            f"    Zernike : R²={quality_z['r2']:.3f}  RMS_res={quality_z['rms_residual']:.2f}  "
            f"dip={dip_amp_z:.2f} θ={dip_angle_z:.1f}°\n"
            f"    Dip NL  : R²={quality_d['r2']:.3f}  RMS_res={quality_d['rms_residual']:.2f}  "
            f"A={dip_amp_d:.2f} θ={dip_angle_d:.1f}° sep={dip_sep_d:.2f}\n"
            f"    Dip LIN : R²={quality_lin['r2']:.3f}  RMS_res={quality_lin['rms_residual']:.2f}  "
            f"A={dip_amp_lin:.2f} θ={dip_angle_lin:.1f}°" + ("  🔴 DIPOLE" if is_dip else "")
        )

    results[band] = band_results
    print(f"  → {len(band_results)} valid decompositions for band {band}")

print("\nDecomposition complete.")

1. **Ce que tes résultats disent vraiment**

Tu as maintenant trois modèles :

(A) Zernike
faible R² (~0.05–0.15)
résidus élevés

- 👉 conclusion : base inadaptée au signal dominant

(B) Dipôle non-linéaire (Gaussian derivative model)
R² parfois bon, parfois catastrophique
instable (sign flips, sep énorme)

- 👉 conclusion : modèle trop rigide + centre mal contraint

(C) Dipôle linéaire (gradient Gaussien)
R² ~0.3 à 0.65 → le meilleur des trois
stable
amplitude cohérente

- 👉 conclusion :

ton signal est très probablement un gradient local de PSF-like profile, pas une décomposition Zernike pure.

2. **Le point clé : ton problème n’est PAS Zernike vs dipôle**

Tu es en train de résoudre implicitement :

❌ hypothèse Zernike (mauvaise ici)
fonction sur disque unité
centre fixe
orthogonalité globale
❌ hypothèse dipôle non-linéaire actuelle
centre libre MAIS optimisation mal contrainte
surface d’erreur très plate → solutions dégénérées
✅ hypothèse qui marche :

- 👉 signal = gradient d’une fonction PSF locale + offset

C’est exactement ton modèle linéaire :

$$I(x,y) \approx A_x \partial_x G + A_y \partial_y G + B$$

3. **Pourquoi tes angles “bougent” encore**

Même si ton fit est meilleur, tu vois :

- angles instables
- amplitude très variable
- parfois rotation de ~180°

- 👉 cause physique :

🔥 forte corrélation Ax / Ay + centre estimé localement

Ton centre est :

```python
x0 = mean(x[valid])
y0 = mean(y[valid])
```

👉 MAIS dans une DIA cutout :

flux asymétrique
blending possible
centroid biaisé

Donc ton dipôle “absorbe” :

shift de centre
asymétrie PSF
résidu de soustraction

4. **Pourquoi Zernike échoue (important)**

Ton intuition initiale était logique mais :

Zernike suppose :
domaine circulaire propre
origine parfaite
orthogonalité globale

Or ici :

cutouts petits (30×30)
objet non centré
gradient dominant
PSF non radiale parfaite

👉 donc :

Zernike ne “voit pas” un dipôle propre → il le dilue dans plein de modes

5. **Pourquoi ton dipôle linéaire marche mieux**

Parce qu’il approxime :

local Taylor expansion de PSF

$$I(x,y) \approx I_0 + \nabla I \cdot (x,y)$$


et ton modèle Gaussien :

encode implicitement la PSF
rend le problème quasi linéaire
robuste au bruit

👉 c’est exactement ce que tu veux pour DIA

6. **Interprétation physique (important pour Rubin / Fink)**

Ton signal ressemble à :

✔️ cas probable
residual de PSF mismatch
centroid shift entre science/template
convolution error
astrometric residual dipole
❌ pas (forcément)
structure physique intrinsèque objet
shapelet higher-order structure
7. Pourquoi tes deux dipoles (NL vs LIN) divergent
NL → optimise centre + amplitude simultanément → dégénérescence
LIN → fixe structure → stable gradient estimate

👉 donc LIN est plus fiable pour classification

8. Conclusion claire
Ton pipeline actuel converge vers :
👉 meilleur modèle = dipole linéaire local (gradient PSF)

Zernike :

bon pour optique globale
mauvais pour cutouts DIA bruités

Dipôle non-linéaire :

trop instable sans contraintes fortes

### Check

In [ ]:
BAND_SEL = "g"
VISINDEX = 1

In [ ]:
band_res = results.get(BAND_SEL, [])

In [ ]:
arr = band_res[VISINDEX]["arr"]
recon = band_res[VISINDEX]["recon_d_lin"]
res = band_res[VISINDEX]["residual_d_lin"]

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)
im1 = ax1.imshow(arr, origin="lower")
im2 = ax2.imshow(recon, origin="lower")
im3 = ax3.imshow(res, origin="lower")
plt.suptitle(" Fit with fit_zernike() ")
ax1.set_title("arr image")
ax2.set_title("recon image")
ax3.set_title("res image")
fig.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04)
fig.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)
fig.colorbar(im3, ax=ax3, fraction=0.046, pad=0.04)
plt.show()

👉 ton code marche parfaitement
👉 ton modèle (Zernike) explique très mal tes données

🔥 1. Le signal clé : R²

Regarde tes valeurs :

$$R^2 \simeq 0.05 rightarrow 0.15$$

👉 Traduction physique :

$$R^2 = 1 - \frac{\sigma^2_{\text{res}}}{\sigma^2_{\text{data}}}$$

Donc ici :

$$ \sigma_{res} \simeq 0.9 \sigma_{data}$$


👉 donc :

90% de ton image reste dans le résidu

🎯 3. **Interprétation physique (très important)**

Ton fit Zernike capture :

- piston (c0)
- tip/tilt (c1, c2)
- quelques modes bas

Mais ton image DIA contient :

- bruit dominant
- structure non radiale
- dipôle pas parfaitement centré
- PSF non modélisée par Zernike

👉 donc :

- Zernike n’est PAS une bonne base pour ces données


## 8. Figure A — Coefficient spectrum overlay (all visits on one plot per band)

Each visit is one line on a `|c_j|` vs Noll-index plot, colour-coded by MJD.
Dominant modes (dipole Z2/Z3, astigmatism Z5/Z6…) immediately stand out as
common peaks across all visits.

In [ ]:
#           "recon_z": recon_z,
#            "residual_z": residual_z,
#            "z_coeffs": coeffs_z,
#            "z_power": power_z,
#            "dip_amp_z": dip_amp_z,
#            "dip_angle_z": dip_angle_z,
#            "quality_z": quality_z,

In [ ]:
jj = np.arange(1, N_ZERN + 1)
xlabels = [ZERNIKE_LABELS.get(j, f"Z{j}") for j in jj]

for band in bands_to_use:
    band_res = results.get(band, [])
    if not band_res:
        continue
    bcolor = BAND_COLORS.get(band, "gray")

    mjds = np.array([r["mjd"] for r in band_res])
    mjd_lo = mjds.min()
    mjd_hi = mjds.max()
    norm = mcolors.Normalize(vmin=mjd_lo, vmax=max(mjd_hi, mjd_lo + 1))
    cmap_v = cm.get_cmap("plasma")

    fig, (ax_abs, ax_signed) = plt.subplots(2, 1, figsize=(12, 7), sharex=True, gridspec_kw={"hspace": 0.35})

    for rec in band_res:
        c = cmap_v(norm(rec["mjd"]))
        lw = 1.8 if rec["is_dipole"] else 0.9
        alpha = 0.95 if rec["is_dipole"] else 0.6
        ls = "-" if rec["is_dipole"] else "--"
        label = f"{rec['date']} {'⬤' if rec['is_dipole'] else ''}"

        ax_abs.semilogy(jj, np.abs(rec["z_coeffs"]) + 1e-6, color=c, lw=lw, ls=ls, alpha=alpha, label=label)
        ax_signed.plot(jj, rec["z_coeffs"], color=c, lw=lw, ls=ls, alpha=alpha)

    # Colourbar for visit date
    sm = cm.ScalarMappable(cmap=cmap_v, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=[ax_abs, ax_signed], fraction=0.025, pad=0.02)
    cbar.set_label("MJD", fontsize=9)

    # Shade morphological groups
    group_spans = [
        (1, 1, "#f0e0ff", "piston"),
        (2, 3, "#ffe0e0", "dipole"),
        (4, 4, "#e0f0ff", "defocus"),
        (5, 6, "#e0ffe0", "astig"),
        (7, 8, "#fff0e0", "coma"),
        (9, 10, "#f0ffe0", "trefoil"),
        (11, 11, "#e8e8ff", "spher"),
    ]
    for j0, j1, fc, lbl in group_spans:
        if j1 > N_ZERN:
            continue
        for ax in (ax_abs, ax_signed):
            ax.axvspan(j0 - 0.5, j1 + 0.5, color=fc, alpha=0.35, zorder=0)
        ax_abs.text(
            (j0 + j1) / 2,
            ax_abs.get_ylim()[0] if ax_abs.get_ylim()[0] > 0 else 1e-4,
            lbl,
            ha="center",
            va="bottom",
            fontsize=7,
            color="#555",
            transform=ax_abs.get_xaxis_transform(),
        )

    ax_abs.set_ylabel("|c_j|  (log scale)", fontsize=9)
    ax_abs.set_title(
        f"Zernike |coefficient| spectrum — band {band} — {len(band_res)} visits", fontsize=10, color=bcolor
    )
    ax_abs.grid(True, which="both", alpha=0.3)
    ax_abs.axhline(0, color="k", lw=0.5, ls=":")

    ax_signed.axhline(0, color="k", lw=0.7, ls=":")
    ax_signed.set_ylabel("c_j  (signed)", fontsize=9)
    ax_signed.set_title("Signed coefficients", fontsize=9)
    ax_signed.grid(True, alpha=0.3)

    ax_signed.set_xticks(jj)
    ax_signed.set_xticklabels(xlabels, fontsize=7, rotation=0)
    ax_signed.set_xlabel("Noll index", fontsize=9)

    # Legend (only dipole-flagged visits to avoid clutter)
    dip_recs = [r for r in band_res if r["is_dipole"]]
    if dip_recs:
        from matplotlib.lines import Line2D

        legend_elements = [
            Line2D([0], [0], color=cmap_v(norm(r["mjd"])), lw=2, label=f"dipole {r['date']}")
            for r in dip_recs
        ]
        ax_abs.legend(
            handles=legend_elements, fontsize=7, loc="upper right", bbox_to_anchor=(1.3, 1.5), framealpha=0.8
        )

    fig.suptitle(
        f"diaObjectId={DIAOBJECT_ID}   band={band}   "
        f"Gaia G={G_str}  status={gaia_status}  group={fink_group}\n"
        f"Decomposition on: {CUTOUT_KIND}   N_Zern={N_ZERN}   "
        f"colour = visit MJD   dashed = non-dipole",
        fontsize=10,
        y=1.01,
    )
    plt.tight_layout()

    if SAVE_FIGS:
        stem = f"zernike_spectrum_obj{DIAOBJECT_ID}_band{band}"
        for ext in ("pdf", "png"):
            plt.savefig(os.path.join(DIR_FIGS, f"{stem}.{ext}"), bbox_inches="tight", dpi=130)
        print(f"→ saved {stem}.{{pdf,png}}")

    plt.show()
    plt.close(fig)

## 9. Figure B — Per-visit coefficient bar chart grid

One subplot per visit.  Each bar chart shows the first `N_DISPLAY` Zernike
coefficients `c_j` with:
- Bar height = `c_j` (signed).
- Bar colour encodes the morphological group.
- The *dipole vector* `(c2, c3)` is indicated as an arrow on a small inset polar plot.


In [ ]:
# Colours per Zernike group (applied to bars by Noll index)
_ZERN_BAR_COLORS = {
    1: "#b39ddb",  # piston — purple
    2: "#ef9a9a",  # tip — red
    3: "#ef9a9a",  # tilt — red
    4: "#90caf9",  # defocus — blue
    5: "#a5d6a7",  # astig — green
    6: "#a5d6a7",
    7: "#ffcc80",  # coma — orange
    8: "#ffcc80",
    9: "#c8e6c9",  # trefoil — light green
    10: "#c8e6c9",
    11: "#b0bec5",  # spherical — grey
}
DEFAULT_BAR_COLOR = "#e0e0e0"


for band in bands_to_use:
    band_res = results.get(band, [])
    if not band_res:
        continue
    bcolor = BAND_COLORS.get(band, "gray")
    n_vis = len(band_res)
    n_cols = min(NCOLS_GRID, n_vis)
    n_rows = int(np.ceil(n_vis / n_cols))

    fig = plt.figure(figsize=(4.5 * n_cols, 4.2 * n_rows))
    fig.suptitle(
        f"Zernike coefficients c₁…c{N_DISPLAY} per visit — diaObjectId={DIAOBJECT_ID}   band={band}\n"
        f"Gaia G={G_str}  status={gaia_status}  group={fink_group}   "
        f"({CUTOUT_KIND} cutout, N_Zern={N_ZERN})",
        fontsize=14,
        color=bcolor,
        fontweight="bold",
        y=1.0,
    )

    for vis_idx, rec in enumerate(band_res):
        row_i = vis_idx // n_cols
        col_i = vis_idx % n_cols
        ax = fig.add_subplot(n_rows, n_cols, vis_idx + 1)

        n_disp = min(N_DISPLAY, len(rec["z_coeffs"]))
        jj_d = np.arange(1, n_disp + 1)
        cc_d = rec["z_coeffs"][:n_disp]
        bar_colors = [_ZERN_BAR_COLORS.get(j, DEFAULT_BAR_COLOR) for j in jj_d]

        bars = ax.bar(jj_d, cc_d, color=bar_colors, edgecolor="k", linewidth=0.5, zorder=3)
        ax.axhline(0, color="k", lw=0.7, ls=":")
        ax.set_xticks(jj_d)
        ax.set_xticklabels(
            [ZERNIKE_LABELS.get(j, f"Z{j}").split("\n")[0] for j in jj_d], fontsize=7, rotation=45, ha="right"
        )
        ax.tick_params(axis="y", labelsize=10)
        ax.grid(True, axis="y", alpha=0.3)

        # Symmetric y-axis across band
        all_band_coeffs = np.concatenate([r["z_coeffs"][:n_disp] for r in band_res])
        yabs = max(float(np.nanmax(np.abs(all_band_coeffs))), 1e-6) * 1.15
        ax.set_ylim(-yabs, yabs)

        # Fit quality annotation
        q = rec["quality_z"]
        dip_str = " 🔴" if rec["is_dipole"] else ""
        title = (
            f"visit {rec['visit']}  {rec['date']}{dip_str}\n"
            f"R²={q['r2']:.3f}  RMS_in={q['rms_input']:.1f}  RMS_res={q['rms_residual']:.1f}\n"
            f"|dip|={rec['dip_amp_z']:.3f}  θ={rec['dip_angle_z']:.1f}°"
        )
        ax.set_title(title, fontsize=7)
        ax.set_ylabel("c_j" if col_i == 0 else "", fontsize=10)

        # Dipole power inset: small arrow showing (c2, c3) direction
        if N_ZERN >= 3:
            c2, c3 = float(rec["z_coeffs"][1]), float(rec["z_coeffs"][2])
            ax_ins = ax.inset_axes([0.75, 0.70, 0.22, 0.28])
            ax_ins.set_aspect("equal")
            ax_ins.set_xlim(-1, 1)
            ax_ins.set_ylim(-1, 1)
            ax_ins.axhline(0, color="#aaa", lw=0.5)
            ax_ins.axvline(0, color="#aaa", lw=0.5)
            amp_norm = np.hypot(c2, c3)
            scale = min(amp_norm / max(yabs / np.sqrt(2), 1e-9), 1.0)
            ax_ins.annotate(
                "",
                xy=(c2 / max(amp_norm, 1e-9) * scale, c3 / max(amp_norm, 1e-9) * scale),
                xytext=(0, 0),
                arrowprops=dict(arrowstyle="->", color="#cc3333", lw=1.5),
            )
            ax_ins.set_xticks([])
            ax_ins.set_yticks([])
            ax_ins.set_title("dip", fontsize=8, pad=1)
            # Unit circle
            circ = plt.Circle((0, 0), 1, color="#ddd", fill=False, lw=0.5)
            ax_ins.add_patch(circ)

    # Hide unused subplots
    total_axes = n_rows * n_cols
    for extra in range(n_vis, total_axes):
        fig.add_subplot(n_rows, n_cols, extra + 1).axis("off")

    plt.tight_layout()

    if SAVE_FIGS:
        stem = f"zernike_bars_obj{DIAOBJECT_ID}_band{band}"
        for ext in ("pdf", "png"):
            plt.savefig(os.path.join(DIR_FIGS, f"{stem}.{ext}"), bbox_inches="tight", dpi=130)
        print(f"→ saved {stem}.{{pdf,png}}")

    plt.show()
    plt.close(fig)

## 10. Figure C — Image triplet grid per visit: original / reconstruction / residual

For each visit, three images side-by-side on the same diverging colour scale:
1. `D` — original DIA difference cutout from the alert stream
2. `D_recon` — Zernike reconstruction (sum of Z1..Z_N_ZERN)
3. `D - D_recon` — residual (what cannot be explained by the first N terms)


In [ ]:
# =================
# Dipole linear (stable)
# =================
#   "recon_d_lin": model_lin,
#   "residual_d_lin": residual_lin,
#   "dip_amp_d_lin": dip_amp_lin,
#   "dip_angle_d_lin": dip_angle_lin,
#   "quality_d_lin": quality_lin,

In [ ]:
CMAP_DIFF = "RdBu_r"

# loop on bands
for band in bands_to_use:
    band_res = results.get(band, [])
    if not band_res:
        continue
    bcolor = BAND_COLORS.get(band, "gray")
    n_vis = len(band_res)

    # Global shared vmax across the band for the original and reconstruction
    all_orig = np.concatenate([r["arr"].ravel() for r in band_res])
    vmax_global = symvlim(all_orig[np.isfinite(all_orig)], DIFF_PERCENTILE)

    # Grid: each visit = one row of 3 panels
    fig, axes = plt.subplots(
        n_vis,
        3,
        figsize=(12, 4.25 * n_vis),
        # squeeze=False,
        gridspec_kw={"hspace": 0.001, "wspace": 0.005},
        layout="constrained",
    )

    # loop on visits
    for vis_idx, rec in enumerate(band_res):
        ax_orig = axes[vis_idx][0]
        ax_recon = axes[vis_idx][1]
        ax_resid = axes[vis_idx][2]

        arr = rec["arr"]
        recon = rec["recon_d_lin"]
        residual = rec["residual_d_lin"]

        # Per-visit residual scale (may be much smaller than global)
        vmax_res = symvlim(residual[np.isfinite(residual)], DIFF_PERCENTILE)

        dip_str = "  🔴 DIPOLE" if rec["is_dipole"] else ""
        row_title = f"visit {rec['visit']}  {rec['date']}{dip_str}   R²={rec['quality_d_lin']['r2']:.3f}"

        for ax, img, vmax_img, title_suffix in [
            (ax_orig, arr, vmax_global, f"Original D\n(±{vmax_global:.1f})"),
            (ax_recon, recon, vmax_global, f"Dipole linear recon\n(±{vmax_global:.1f})"),
            (ax_resid, residual, vmax_res, f"Residual D – recon\n(±{vmax_res:.1f})"),
        ]:
            disp = np.where(np.isfinite(img), img, 0.0)
            im = ax.imshow(disp, origin="lower", cmap=CMAP_DIFF, vmin=-vmax_img, vmax=vmax_img)
            plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            draw_crosshair(ax, disp.shape)
            ax.axis("off")
            ax.set_title(title_suffix, fontsize=8)

        # Row label on the left
        ax_orig.set_ylabel(row_title, fontsize=7.5, labelpad=4)
        ax_orig.axis("on")
        ax_orig.set_yticks([])
        ax_orig.set_xticks([])
        for sp in ax_orig.spines.values():
            sp.set_visible(False)

    fig.suptitle(
        # f"Zernike image triplet — diaObjectId={DIAOBJECT_ID}   band={band}\n"
        f"Dipole linear Fit — diaObjectId={DIAOBJECT_ID}   band={band}\n"
        f"Gaia G={G_str}  status={gaia_status}  group={fink_group}   "
        f"cutout={CUTOUT_KIND}   N_Zern={N_ZERN}",
        fontsize=10,
        color=bcolor,
        fontweight="bold",
        y=1.00,
    )
    # plt.tight_layout()

    if SAVE_FIGS:
        # stem = f"zernike_triplet_obj{DIAOBJECT_ID}_band{band}"
        stem = f"dipole_linear_fit_obj{DIAOBJECT_ID}_band{band}"
        for ext in ("pdf", "png"):
            plt.savefig(os.path.join(DIR_FIGS, f"{stem}.{ext}"), bbox_inches="tight", dpi=130)
        print(f"→ saved {stem}.{{pdf,png}}")

    plt.show()
    plt.close(fig)

## 11. Figure D — Dipole amplitude & angle evolution across visits

Two panels:
- **Top** : dipole amplitude `|dip| = sqrt(c2² + c3²)` vs MJD, per band.
- **Bottom** : dipole angle `θ = atan2(c3, c2)` vs MJD.

If the dipole originates from a rigid rotation of the PSF template relative to
the science image (astrometric rotation), the angle should correlate with the
parallactic angle or the field rotation between visits.

In [ ]:
# =================
# Dipole linear (stable)
# =================
#            "recon_d_lin": model_lin,
#            "residual_d_lin": residual_lin,
#            "dip_amp_d_lin": dip_amp_lin,
#            "dip_angle_d_lin": dip_angle_lin,
#            "quality_d_lin": quality_lin,

In [ ]:
# "dip_amp_d_lin": dip_amp_lin,
# "dip_angle_d_lin": dip_angle_lin,

In [ ]:
fig, (ax_amp, ax_ang) = plt.subplots(2, 1, figsize=(11, 6), sharex=False)

# loop on bands
for band in bands_to_use:
    band_res = results.get(band, [])
    if not band_res:
        continue
    bcolor = BAND_COLORS.get(band, "gray")
    mjds = np.array([r["mjd"] for r in band_res])
    amps = np.array([r["dip_amp"] for r in band_res])
    angles = np.array([r["dip_angle"] for r in band_res])
    is_dip = np.array([r["is_dipole"] for r in band_res])

    # Compute Δt from first detection
    t0 = mjds.min()
    dt = mjds - t0

    ax_amp.scatter(dt[~is_dip], amps[~is_dip], color=bcolor, marker="o", s=40, alpha=0.7, label=f"{band}")
    ax_amp.scatter(
        dt[is_dip],
        amps[is_dip],
        color=bcolor,
        marker="*",
        s=180,
        edgecolors="k",
        lw=0.8,
        alpha=0.95,
        label=f"{band} dipole",
    )
    ax_amp.plot(dt, amps, color=bcolor, lw=0.8, alpha=0.5)

    ax_ang.scatter(dt[~is_dip], angles[~is_dip], color=bcolor, marker="o", s=40, alpha=0.7)
    ax_ang.scatter(
        dt[is_dip], angles[is_dip], color=bcolor, marker="*", s=180, edgecolors="k", lw=0.8, alpha=0.95
    )
    ax_ang.plot(dt, angles, color=bcolor, lw=0.8, alpha=0.5)

ax_amp.axhline(0, color="k", lw=0.5, ls=":")
ax_amp.set_ylabel("Dipole amplitude  √(c₂² + c₃²)", fontsize=9)
ax_amp.set_title("Zernike dipole amplitude vs time", fontsize=10)
ax_amp.legend(fontsize=8, ncol=4, framealpha=0.8)
ax_amp.grid(True, alpha=0.3)

ax_ang.axhline(0, color="k", lw=0.5, ls=":")
ax_ang.set_ylabel("Dipole angle  atan2(c₃, c₂)  [°]", fontsize=9)
ax_ang.set_xlabel("Δt [days from first detection]", fontsize=9)
ax_ang.set_title("Zernike dipole angle vs time", fontsize=10)
ax_ang.set_ylim(-185, 185)
ax_ang.axhline(-180, color="#aaa", lw=0.5, ls="--")
ax_ang.axhline(180, color="#aaa", lw=0.5, ls="--")
ax_ang.grid(True, alpha=0.3)

fig.suptitle(
    f"diaObjectId={DIAOBJECT_ID}   Gaia G={G_str}  status={gaia_status}  group={fink_group}\n"
    f"Zernike dipole evolution   cutout={CUTOUT_KIND}   ★ = isDipole flag",
    fontsize=10,
    y=1.01,
)
plt.tight_layout()

if SAVE_FIGS:
    stem = f"zernike_dipole_evolution_obj{DIAOBJECT_ID}"
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{stem}.{ext}"), bbox_inches="tight", dpi=130)
    print(f"→ saved {stem}.{{pdf,png}}")

plt.show()
plt.close(fig)

## 12. Figure E — Group-power pie / bar per band

Bar chart of total power `Σ c_j²` per morphological group (piston, dipole,
defocus, astigmatism, coma…), summed over all visits in the band.  Reveals
which mode family dominates the alert-stream residuals.

In [ ]:
GROUP_COLORS = {
    "Z1_piston": "#b39ddb",
    "Z2-3_dipole": "#ef5350",
    "Z4_defocus": "#42a5f5",
    "Z5-6_astig": "#66bb6a",
    "Z7-8_coma": "#ffa726",
    "Z9-10_trefoil": "#26c6da",
    "Z11_spher": "#8d6e63",
    "Z12+_higher": "#bdbdbd",
}

bands_with_results = [b for b in bands_to_use if results.get(b)]
n_bands = len(bands_with_results)
if n_bands == 0:
    print("No results to display.")
else:
    fig, axes = plt.subplots(1, n_bands, figsize=(4.5 * n_bands, 5), squeeze=False)

    for ax_i, band in enumerate(bands_with_results):
        band_res = results[band]
        bcolor = BAND_COLORS.get(band, "gray")
        ax = axes[0][ax_i]

        # Accumulate group power over all visits
        group_names = list(GROUP_COLORS.keys())
        total_power = {g: 0.0 for g in group_names}
        for rec in band_res:
            for g, v in rec["power"].items():
                if g in total_power:
                    total_power[g] += v

        powers = np.array([total_power[g] for g in group_names])
        total = powers.sum()
        fracs = 100 * powers / max(total, 1e-30)
        colors = [GROUP_COLORS[g] for g in group_names]
        short_names = [g.split("_")[1] for g in group_names]  # e.g. "piston", "dipole"

        bars = ax.bar(short_names, fracs, color=colors, edgecolor="k", linewidth=0.6)
        ax.set_ylabel("Power fraction [%]" if ax_i == 0 else "", fontsize=9)
        ax.set_ylim(0, max(fracs) * 1.25)
        ax.set_xticklabels(short_names, rotation=35, ha="right", fontsize=8)
        ax.grid(True, axis="y", alpha=0.3)
        ax.set_title(f"Band {band}\n{len(band_res)} visits", fontsize=10, color=bcolor, fontweight="bold")

        # Annotate percentage on top of each bar
        for bar, frac in zip(bars, fracs):
            if frac > 1.0:
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.5,
                    f"{frac:.1f}%",
                    ha="center",
                    va="bottom",
                    fontsize=7,
                )

    fig.suptitle(
        f"Zernike group power — diaObjectId={DIAOBJECT_ID}\n"
        f"Gaia G={G_str}  status={gaia_status}  group={fink_group}   "
        f"cutout={CUTOUT_KIND}   N_Zern={N_ZERN}",
        fontsize=10,
        y=1.02,
    )
    plt.tight_layout()

    if SAVE_FIGS:
        stem = f"zernike_group_power_obj{DIAOBJECT_ID}"
        for ext in ("pdf", "png"):
            plt.savefig(os.path.join(DIR_FIGS, f"{stem}.{ext}"), bbox_inches="tight", dpi=130)
        print(f"→ saved {stem}.{{pdf,png}}")

    plt.show()
    plt.close(fig)

## 13. Quantitative summary table

In [ ]:
from IPython.display import display as ipy_display

rows = []
for band in bands_to_use:
    for rec in results.get(band, []):
        q = rec["quality"]
        row = {
            "band": band,
            "visit": rec["visit"],
            "date": rec["date"],
            "isDipole": rec["is_dipole"],
            "psfFlux_nJy": round(rec["psfFlux"], 2),
            "snr": round(rec["snr"], 2) if np.isfinite(rec["snr"]) else np.nan,
            "dip_amp": round(rec["dip_amp"], 4),
            "dip_angle_deg": round(rec["dip_angle"], 2),
            "R2_fit": round(q["r2"], 4),
            "rms_input": round(q["rms_input"], 4),
            "rms_residual": round(q["rms_residual"], 4),
            "n_fit_pixels": q["n_fit_pixels"],
        }
        # Add individual coefficients c1..cN_DISPLAY
        for j in range(1, min(N_DISPLAY, len(rec["coeffs"])) + 1):
            row[f"c{j}"] = round(float(rec["coeffs"][j - 1]), 5)
        rows.append(row)

df_summary = pd.DataFrame(rows)
print(f"Zernike decomposition summary — diaObjectId={DIAOBJECT_ID}")
ipy_display(df_summary)

csv_path = os.path.join(DIR_FIGS, f"zernike_summary_obj{DIAOBJECT_ID}.csv")
df_summary.to_csv(csv_path, index=False)
print(f"\n→ Saved to {csv_path}")